# Origin sweep for approach B (fast): which origin forecasts the latest real days best?

**Why this exists:** the full forecast takes about 5 hours. This notebook answers one diagnostic question in minutes by changing three things:
- **2,000 sampled series** instead of all of them (same series for every origin, so the comparison is paired);
- **one forward pass** per origin using the model's distribution parameters (`predict_likelihood_parameters`), instead of 100 samples per series;
- **GPU** if one is available (the full script forces `map_location="cpu"`).

**Question it answers:** forecast from several origins (default 10, 17, 24 and 31 Oct), score every origin on the same real days, and see whether the 31 Oct history is the problem (late-posted sales) or whether every origin under-forecasts the days before Diwali (a model or flag problem).

**Needs:** the widened data refresh (`01b_data_refresh_widened.ipynb`, ORIGIN = last complete actual day, which includes the real 1-3 Nov sales) and `calendar_extended_to_dec31.parquet`. Run from the same folder as the other notebooks.

In [ ]:
import os, json, glob, gc
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import TFTModel
from pytorch_lightning import Callback

BASE = os.getcwd()
MODEL_NAME   = "daily_tft_negbin_emb_2026-10-09_11_37_06"        # the approach-B model
ACTUALS_DIR  = os.path.join(BASE, "local_train_data_refresh")     # from 01b_data_refresh_widened
CALENDAR_PATH = os.path.join(BASE, "calendar_extended_to_dec31.parquet")
OUT_DIR      = os.path.join(BASE, "origin_sweep")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- settings ----------------
ORIGINS      = ["2026-10-10", "2026-10-17", "2026-10-24", "2026-10-31"]      # every origin must be before ACT_END
ACT_END      = "2026-11-03"          # last day with COMPLETE actuals (see the completeness table below before trusting it)
FORECAST_END = pd.Timestamp("2026-12-31")
N_SAMPLE     = 2000
SEED         = 0
BATCH_SIZE   = 512
BASELINE_WEEKS = 4                    # naive benchmark: average of the same weekday over this many weeks before the origin

In [ ]:
with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)
with open(os.path.join(BASE, ROLES["vocab_path"])) as f:
    VOCAB = json.load(f)
time_col, group_col, target_col, FREQ = ROLES["time_col"], ROLES["group_col"], ROLES["target_col"], ROLES["freq"]
static = ROLES["static_covariates"]
fut    = ROLES["future_covariates"]
ICL, OCL = ROLES["input_chunk_length"], ROLES["output_chunk_length"]
ORIGINS = [pd.Timestamp(o) for o in ORIGINS]
ACT_END = pd.Timestamp(ACT_END)
EARLIEST_START = min(ORIGINS) - pd.Timedelta(days=ICL - 1)

try:
    import tft_checks  # noqa: F401
except ImportError:
    pass
class TBDiagnostics(Callback):
    def __init__(self, *a, **k): pass

# ---------------- choose the sample of series (cheap: key column only) ----------------
files = sorted(glob.glob(os.path.join(ACTUALS_DIR, "chunk_*.parquet")))
assert files, f"No chunk_*.parquet in {ACTUALS_DIR}; run 01b_data_refresh_widened first."
all_keys = pd.concat([pd.read_parquet(p, columns=[group_col]) for p in files])[group_col].astype(str).drop_duplicates()
keys = all_keys.sample(n=min(N_SAMPLE, len(all_keys)), random_state=SEED).tolist()
keyset = set(keys)
print(f"{len(all_keys):,} series in the refresh; using {len(keys):,}")

# ---------------- load their data once ----------------
parts = []
for p in files:
    df = pd.read_parquet(p, columns=[time_col, group_col, target_col] + static)
    df[group_col] = df[group_col].astype(str)
    df = df[df[group_col].isin(keyset)]
    if len(df):
        df[time_col] = pd.to_datetime(df[time_col])
        parts.append(df[df[time_col] >= EARLIEST_START])
DATA = pd.concat(parts, ignore_index=True)
del parts; gc.collect()
print(f"Data {DATA[time_col].min().date()} -> {DATA[time_col].max().date()} | rows {len(DATA):,}")
assert DATA[time_col].max() >= ACT_END, f"data ends {DATA[time_col].max().date()}, before ACT_END {ACT_END.date()}"
assert DATA[time_col].min() <= EARLIEST_START, (
    f"data starts {DATA[time_col].min().date()} but the earliest origin needs {EARLIEST_START.date()}; widen the refresh (EARLIEST_ORIGIN).")

# ---------------- matrix: series x days ----------------
DAYS = pd.date_range(EARLIEST_START, ACT_END, freq=FREQ)
row = {k: i for i, k in enumerate(keys)}
Y = np.full((len(keys), len(DAYS)), np.nan)
Y[DATA[group_col].map(row).to_numpy(), (DATA[time_col] - DAYS[0]).dt.days.to_numpy()] = DATA[target_col].to_numpy(dtype=float)
ok = ~np.isnan(Y).any(axis=1)
print(f"Series complete on every day {DAYS[0].date()} -> {DAYS[-1].date()}: {int(ok.sum()):,} of {len(keys):,}")
assert ok.sum() > 0
STAT = DATA.drop_duplicates(group_col).set_index(group_col).reindex(keys)[static]
code_maps = {c: {v: i for i, v in enumerate(VOCAB[c])} for c in static}
CODES = np.array([[code_maps[c].get(str(STAT.iloc[i][c]), 0) for c in static] for i in range(len(keys))], dtype=np.float32)
keep = np.where(ok)[0]
Y, CODES, keys = Y[keep], CODES[keep], [keys[i] for i in keep]
print(f"Using {len(keys):,} complete series")

### Is the recent data complete? (look before trusting the last days)

In [ ]:
tot = pd.Series(Y.sum(axis=0), index=DAYS)
zero_share = pd.Series((Y == 0).mean(axis=0), index=DAYS)
chk = pd.DataFrame({"total sales (sample)": tot, "share of series with 0 sales": zero_share})
chk["same weekday, 7 days earlier"] = tot.shift(7)
chk["ratio to 7 days earlier"] = tot / tot.shift(7)
pd.set_option("display.width", 220); pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print(chk.tail(14).to_string())
print("\nA recent day whose total is far below its weekday a week earlier, with a higher share of zero-sales series, "
      "suggests late-posted sales. Compare against the festival pattern before deciding.")

In [ ]:
# ---------------- calendar ----------------
def pad_calendar(cal, last_needed):
    last = cal[time_col].max()
    if last >= last_needed:
        return cal
    idx = pd.date_range(last + pd.Timedelta(days=1), last_needed, freq="D")
    pad = pd.DataFrame(0.0, index=range(len(idx)), columns=fut)
    pad.insert(0, time_col, idx)
    dt = pad[time_col].dt
    if "DOW_SIN" in fut: pad["DOW_SIN"] = np.sin(2 * np.pi * dt.dayofweek / 7)
    if "DOW_COS" in fut: pad["DOW_COS"] = np.cos(2 * np.pi * dt.dayofweek / 7)
    if "IS_MONTH_END" in fut: pad["IS_MONTH_END"] = (dt.day >= dt.days_in_month - 2).astype(float)
    if "IS_MONTH_START" in fut: pad["IS_MONTH_START"] = (dt.day <= 3).astype(float)
    if "DAYS_TO_MONTH_END" in fut: pad["DAYS_TO_MONTH_END"] = (dt.days_in_month - dt.day) / 31.0
    return pd.concat([cal, pad], ignore_index=True)

cal = pd.read_parquet(CALENDAR_PATH)
cal[time_col] = pd.to_datetime(cal[time_col])
cal = cal.sort_values(time_col).reset_index(drop=True)
assert cal[time_col].min() <= EARLIEST_START and cal[time_col].max() >= FORECAST_END
assert not cal[fut].isna().any().any()
cal = pad_calendar(cal, max(ORIGINS) + pd.Timedelta(days=OCL))      # Darts needs covariates to origin + output_chunk_length
COV = TimeSeries.from_dataframe(cal, time_col=time_col, value_cols=fut, freq=FREQ).astype(np.float32)

model = TFTModel.load_from_checkpoint(MODEL_NAME, best=True)      # GPU if available
print(f"Model loaded | device: {'GPU' if torch.cuda.is_available() else 'CPU'} | input {ICL}, output {OCL}")

EXPECTED = [f"{target_col}_r", f"{target_col}_p"]
def forecast_from(origin):
    n = min((FORECAST_END - origin).days, OCL)
    start = origin - pd.Timedelta(days=ICL - 1)
    i0, i1 = (start - DAYS[0]).days, (origin - DAYS[0]).days
    hist = [TimeSeries.from_times_and_values(DAYS[i0:i1 + 1], Y[i, i0:i1 + 1].reshape(-1, 1).astype(np.float32),
                                             columns=[target_col],
                                             static_covariates=pd.DataFrame([CODES[i]], columns=static, dtype=np.float32))
            for i in range(len(keys))]
    preds = model.predict(n=n, series=hist, future_covariates=[COV] * len(hist),
                          predict_likelihood_parameters=True, num_samples=1, batch_size=BATCH_SIZE, verbose=False)
    out = np.zeros((len(preds), n))
    for i, p in enumerate(preds):
        assert list(p.components) == EXPECTED, list(p.components)
        v = p.values(copy=False).astype(np.float64)
        r, pr = v[:, 0], v[:, 1]
        assert np.isfinite(v).all() and (r > 0).all() and ((pr > 0) & (pr < 1)).all()
        out[i] = r * pr / (1.0 - pr)
    return pd.date_range(origin + pd.Timedelta(days=1), periods=n, freq=FREQ), out

def baseline_from(origin, days):
    """Average of the same weekday over the BASELINE_WEEKS weeks before the origin."""
    out = np.zeros((len(keys), len(days)))
    for j, d in enumerate(days):
        lags = [d - pd.Timedelta(days=7 * k) for k in range(1, BASELINE_WEEKS + 1)]
        lags = [l for l in lags if l <= origin]
        # a lag counts only if it is on or before the origin; walk back further if needed
        k = 1
        while len(lags) < BASELINE_WEEKS:
            cand = d - pd.Timedelta(days=7 * (BASELINE_WEEKS + k)); k += 1
            if cand <= origin: lags.append(cand)
        out[:, j] = np.mean([Y[:, (l - DAYS[0]).days] for l in lags], axis=0)
    return out

FC, t0 = {}, datetime.now()
for o in ORIGINS:
    days, f = forecast_from(o)
    FC[o] = (days, f)
    print(f"origin {o.date()}: forecast {days[0].date()} -> {days[-1].date()} ({len(days)} days)   {(datetime.now() - t0).seconds}s elapsed")

In [ ]:
# ---------------- score every origin on the SAME real days ----------------
scored_origins = [o for o in ORIGINS if o < ACT_END]
assert scored_origins, 'every origin must be earlier than ACT_END so there are real days to score'
first_common = max(scored_origins) + pd.Timedelta(days=1)
common = pd.date_range(first_common, ACT_END, freq=FREQ)
print(f"Common scored days (after the latest origin): {common[0].date() if len(common) else 'none'} -> {common[-1].date() if len(common) else ''} ({len(common)} days)")

def cols(days):
    return [(d - DAYS[0]).days for d in days]

def metrics(F, A):
    fs, as_ = F.sum(axis=1), A.sum(axis=1)
    return {"forecast / actual": F.sum() / A.sum(),
            "WMAPE daily total": np.abs(F.sum(0) - A.sum(0)).sum() / A.sum(),
            "WMAPE per series total": np.abs(fs - as_).sum() / A.sum()}

rows = []
# (a) each origin on its OWN days up to ACT_END
for o in scored_origins:
    days, f = FC[o]
    m = days <= ACT_END
    d = days[m]
    A = Y[:, cols(d)]
    r = {"origin": o.date(), "scored on": f"{d[0].date()}..{d[-1].date()}", "days": len(d)}
    r.update(metrics(f[:, m], A))
    b = baseline_from(o, d)
    r["baseline WMAPE daily total"] = np.abs(b.sum(0) - A.sum(0)).sum() / A.sum()
    r["baseline forecast / actual"] = b.sum() / A.sum()
    rows.append(r)
own = pd.DataFrame(rows).set_index("origin")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print("EACH ORIGIN ON ITS OWN DAYS (up to the last real day)\n")
print(own.to_string())

# (b) every origin on the common days (the fair comparison)
if len(common):
    A = Y[:, cols(common)]
    rows = []
    for o in scored_origins:
        days, f = FC[o]
        sel_ = [list(days).index(d) for d in common if d in days]
        if len(sel_) != len(common):
            continue
        r = {"origin": o.date(), "horizon of first scored day": (common[0] - o).days}
        r.update(metrics(f[:, sel_], A))
        rows.append(r)
    comm = pd.DataFrame(rows).set_index("origin")
    print(f"\nEVERY ORIGIN ON THE SAME DAYS ({common[0].date()} -> {common[-1].date()})\n")
    print(comm.to_string())
    comm.to_csv(os.path.join(OUT_DIR, "origin_sweep_common_days.csv"))
own.to_csv(os.path.join(OUT_DIR, "origin_sweep_own_days.csv"))

# ---------------- daily detail for the last real days ----------------
detail = pd.DataFrame({"ACTUAL": pd.Series(Y.sum(axis=0), index=DAYS)})
for o in ORIGINS:
    days, f = FC[o]
    detail[f"from {o.date()}"] = pd.Series(f.sum(axis=0), index=days)
detail = detail.loc[ACT_END - pd.Timedelta(days=13):ACT_END]
print("\nDAILY TOTALS, LAST 14 REAL DAYS (sample of series)\n")
print(detail.round(0).to_string())
detail.to_csv(os.path.join(OUT_DIR, "origin_sweep_daily_last14.csv"))

fig, ax = plt.subplots(figsize=(12, 5))
x = detail.index
ax.plot(x, detail["ACTUAL"], color="black", lw=2.6, label="ACTUAL")
for c in detail.columns[1:]:
    ax.plot(x, detail[c], lw=1.3, marker="o", ms=3, label=c)
ax.set_title("Forecasts from different origins vs real sales (sample of series)")
ax.legend(fontsize=8); fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "origin_sweep.png"), dpi=130); plt.show()
print("Saved to", OUT_DIR)